## Example 1: Basic Dataset Generation

In [ ]:
from faker import Faker
import faker_datasets
import pandas as pd
import numpy as np
import json
import os
import tempfile

# Example 1: Generate basic sample datasets for data science
fake = Faker()

print("=== Example 1: Basic Dataset Generation ===")

try:
    # faker_datasets turns a JSON file into Faker methods: add_dataset loads the file when the class is
    # defined, `picker=` adds a method that returns a random entry, and with_datasets injects the data
    cities = [
        {"name": "Tokyo", "country": "Japan", "population": 37_400_000},
        {"name": "Lagos", "country": "Nigeria", "population": 15_400_000},
        {"name": "Reykjavik", "country": "Iceland", "population": 140_000},
        {"name": "Montevideo", "country": "Uruguay", "population": 1_380_000},
        {"name": "Tromso", "country": "Norway", "population": 78_000},
    ]
    with tempfile.TemporaryDirectory() as tmpdir:
        cities_path = os.path.join(tmpdir, "cities.json")
        with open(cities_path, "w", encoding="utf-8") as fh:
            json.dump(cities, fh)

        @faker_datasets.add_dataset("cities", cities_path, picker="city_record")
        class CityProvider(faker_datasets.Provider):
            @faker_datasets.with_datasets("cities")
            def big_city(self, cities):
                return faker_datasets.pick(self, cities, match=lambda c: c["population"] > 1_000_000)

    fake.add_provider(CityProvider)

    picked = [fake.city_record() for _ in range(20)]
    assert all(c in cities for c in picked), picked
    big = [fake.big_city() for _ in range(20)]
    assert all(c in cities and c["population"] > 1_000_000 for c in big), big
    print(f"Random dataset entries: {[c['name'] for c in picked[:5]]}")
    print(f"Filtered entries (population > 1M): {sorted({c['name'] for c in big})}")

    # Generate sample dataset for machine learning, with a city column drawn from the dataset
    sample_data = []
    
    for i in range(100):
        row = {
            'id': i + 1,
            'feature_1': fake.pyfloat(left_digits=2, right_digits=4, positive=True),
            'feature_2': fake.pyint(min_value=0, max_value=100),
            'category': fake.random_element(elements=('A', 'B', 'C', 'D')),
            'city': fake.city_record()['name'],
            'target': fake.pybool()
        }
        sample_data.append(row)
    
    # Convert to DataFrame
    df = pd.DataFrame(sample_data)
    assert df.shape == (100, 6) and set(df['city']) <= {c['name'] for c in cities}, df.shape
    
    print(f"Generated dataset shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Data types:\n{df.dtypes}")
    print(f"\nFirst 5 rows:")
    print(df.head())
    
    # Basic statistics
    print(f"\nBasic statistics:")
    print(df.describe())
    
except Exception as e:
    print(f"Error in dataset generation: {e}")
    raise AssertionError(
        f"faker_datasets example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed")

## Example 2: Time Series Dataset Generation

In [ ]:
from faker import Faker
import faker_datasets
import pandas as pd
from datetime import datetime, timedelta
import random

# Example 2: Generate time series dataset for analysis
fake = Faker()

print("=== Example 2: Time Series Dataset Generation ===")

try:
    # Generate time series data for stock prices or metrics
    start_date = datetime(2023, 1, 1)
    end_date = datetime(2024, 1, 1)
    current_date = start_date
    
    time_series_data = []
    base_value = 100.0
    
    while current_date < end_date:
        # Simulate realistic time series with trend and noise
        daily_change = random.uniform(-5, 5) + random.normalvariate(0, 2)
        base_value = max(10, base_value + daily_change)
        
        row = {
            'date': current_date,
            'value': round(base_value, 2),
            'volume': fake.random_int(1000, 50000),
            'category': fake.random_element(elements=['tech', 'finance', 'healthcare']),
            'region': fake.country_code()
        }
        time_series_data.append(row)
        current_date += timedelta(days=1)
    
    # Convert to DataFrame
    ts_df = pd.DataFrame(time_series_data)
    
    print(f"Generated time series shape: {ts_df.shape}")
    print(f"Date range: {ts_df['date'].min()} to {ts_df['date'].max()}")
    print(f"\nFirst 5 rows:")
    print(ts_df.head())
    
    # Time series statistics
    print(f"\nValue statistics:")
    print(f"Min: {ts_df['value'].min():.2f}")
    print(f"Max: {ts_df['value'].max():.2f}")
    print(f"Mean: {ts_df['value'].mean():.2f}")
    print(f"Std: {ts_df['value'].std():.2f}")
    
    # Monthly aggregation
    ts_df['month'] = ts_df['date'].dt.to_period('M')
    monthly_stats = ts_df.groupby('month')['value'].agg(['mean', 'min', 'max'])
    print(f"\nMonthly statistics (first 5 months):")
    print(monthly_stats.head())
    
except Exception as e:
    print(f"Error in time series generation: {e}")
    raise AssertionError(
        f"faker_datasets example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 completed")

## Example 3: Classification Dataset Generation

In [ ]:
from faker import Faker
import faker_datasets
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification

# Example 3: Generate classification datasets for machine learning
fake = Faker()

print("=== Example 3: Classification Dataset Generation ===")

try:
    # Generate synthetic classification dataset
    X, y = make_classification(
        n_samples=1000,
        n_features=10,
        n_informative=5,
        n_redundant=2,
        n_clusters_per_class=1,
        random_state=42
    )
    
    # Create DataFrame with meaningful feature names
    feature_names = [
        'age', 'income', 'education_years', 'work_experience',
        'credit_score', 'debt_ratio', 'savings', 'monthly_spending',
        'loan_history', 'employment_stability'
    ]
    
    # Create comprehensive dataset
    classification_data = pd.DataFrame(X, columns=feature_names)
    classification_data['target'] = y
    
    # Add categorical features using faker
    classification_data['city'] = [fake.city() for _ in range(1000)]
    classification_data['occupation'] = [
        fake.random_element(elements=[
            'Engineer', 'Teacher', 'Doctor', 'Manager', 'Analyst',
            'Developer', 'Consultant', 'Sales', 'Marketing', 'Finance'
        ]) for _ in range(1000)
    ]
    classification_data['education_level'] = [
        fake.random_element(elements=[
            'High School', 'Bachelor', 'Master', 'PhD', 'Associate'
        ]) for _ in range(1000)
    ]
    
    print(f"Generated classification dataset shape: {classification_data.shape}")
    print(f"Features: {list(classification_data.columns)}")
    print(f"\nTarget distribution:")
    print(classification_data['target'].value_counts())
    
    # Show correlation with target
    correlations = classification_data.select_dtypes(include=[np.number]).corrwith(
        classification_data['target']
    ).sort_values(key=abs, ascending=False)
    
    print(f"\nFeature correlations with target:")
    print(correlations.head(8))
    
    # Sample data preview
    print(f"\nSample data (first 3 rows):")
    print(classification_data.head(3))
    
    # Class balance analysis
    class_0_count = sum(classification_data['target'] == 0)
    class_1_count = sum(classification_data['target'] == 1)
    balance_ratio = min(class_0_count, class_1_count) / max(class_0_count, class_1_count)
    
    print(f"\nClass balance analysis:")
    print(f"Class 0: {class_0_count} samples")
    print(f"Class 1: {class_1_count} samples")
    print(f"Balance ratio: {balance_ratio:.3f}")
    
except Exception as e:
    print(f"Error in classification dataset generation: {e}")
    raise AssertionError(
        f"faker_datasets example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 completed")

## Example 4: Regression Dataset with Faker Integration

In [ ]:
from faker import Faker
import faker_datasets
import pandas as pd
import numpy as np
from sklearn.datasets import make_regression

# Example 4: Generate regression datasets for predictive modeling
fake = Faker()

print("=== Example 4: Regression Dataset with Faker Integration ===")

try:
    # Generate synthetic regression dataset
    X, y = make_regression(
        n_samples=500,
        n_features=8,
        noise=10,
        random_state=42
    )
    
    # Create meaningful feature names for house price prediction
    feature_names = [
        'square_footage', 'bedrooms', 'bathrooms', 'garage_spaces',
        'lot_size', 'age', 'school_rating', 'crime_index'
    ]
    
    # Create comprehensive real estate dataset
    regression_data = pd.DataFrame(X, columns=feature_names)
    
    # Transform features to realistic scales
    regression_data['square_footage'] = np.abs(regression_data['square_footage']) * 500 + 800
    regression_data['bedrooms'] = np.abs(regression_data['bedrooms']) % 6 + 1
    regression_data['bathrooms'] = np.abs(regression_data['bathrooms']) % 4 + 1
    regression_data['garage_spaces'] = np.abs(regression_data['garage_spaces']) % 4
    regression_data['lot_size'] = np.abs(regression_data['lot_size']) * 2000 + 5000
    regression_data['age'] = np.abs(regression_data['age']) % 50
    regression_data['school_rating'] = np.abs(regression_data['school_rating']) % 10 + 1
    regression_data['crime_index'] = np.abs(regression_data['crime_index']) % 100
    
    # Scale target to realistic house prices
    regression_data['price'] = np.abs(y) * 50000 + 150000
    
    # Add categorical features using faker
    regression_data['address'] = [fake.address().replace('\n', ', ') for _ in range(500)]
    regression_data['neighborhood'] = [fake.city_suffix() for _ in range(500)]
    regression_data['property_type'] = [
        fake.random_element(elements=[
            'Single Family', 'Condo', 'Townhouse', 'Duplex', 'Ranch'
        ]) for _ in range(500)
    ]
    regression_data['listing_agent'] = [fake.name() for _ in range(500)]
    
    print(f"Generated regression dataset shape: {regression_data.shape}")
    print(f"Features: {list(regression_data.columns)}")
    
    # Target statistics
    print(f"\nPrice statistics:")
    print(f"Min: ${regression_data['price'].min():,.2f}")
    print(f"Max: ${regression_data['price'].max():,.2f}")
    print(f"Mean: ${regression_data['price'].mean():,.2f}")
    print(f"Median: ${regression_data['price'].median():,.2f}")
    
    # Feature correlations with price
    numeric_features = regression_data.select_dtypes(include=[np.number])
    correlations = numeric_features.corrwith(regression_data['price']).sort_values(key=abs, ascending=False)
    
    print(f"\nFeature correlations with price:")
    print(correlations.head(6))
    
    # Sample data preview
    print(f"\nSample properties (first 2 rows):")
    sample_cols = ['square_footage', 'bedrooms', 'bathrooms', 'price', 'neighborhood', 'property_type']
    print(regression_data[sample_cols].head(2))
    
    # Price distribution analysis
    price_ranges = pd.cut(regression_data['price'], bins=5, labels=[
        'Budget', 'Affordable', 'Mid-range', 'Premium', 'Luxury'
    ])
    print(f"\nPrice distribution by range:")
    print(price_ranges.value_counts())
    
except Exception as e:
    print(f"Error in regression dataset generation: {e}")
    raise AssertionError(
        f"faker_datasets example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 completed")

## Example 5: Advanced Multi-Domain Dataset Factory

In [ ]:
from faker import Faker
import faker_datasets
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta
import random

# Example 5: Advanced multi-domain dataset factory for comprehensive testing
fake = Faker()

print("=== Example 5: Advanced Multi-Domain Dataset Factory ===")

class DatasetFactory:
    """Factory for generating domain-specific datasets for data science applications."""
    
    def __init__(self):
        self.fake = Faker()
        self.domains = ['ecommerce', 'healthcare', 'finance', 'social_media']
    
    def generate_ecommerce_dataset(self, n_samples=200):
        """Generate e-commerce transaction dataset."""
        data = []
        for i in range(n_samples):
            row = {
                'transaction_id': self.fake.uuid4()[:8],
                'customer_id': self.fake.uuid4()[:6],
                'product_name': self.fake.catch_phrase(),
                'category': self.fake.random_element(elements=[
                    'Electronics', 'Clothing', 'Books', 'Home', 'Sports'
                ]),
                'price': round(self.fake.pyfloat(left_digits=2, right_digits=2, positive=True) * 100, 2),
                'quantity': self.fake.random_int(1, 5),
                'purchase_date': self.fake.date_between(start_date='-1y', end_date='today'),
                'customer_age': self.fake.random_int(18, 70),
                'customer_city': self.fake.city(),
                'payment_method': self.fake.random_element(elements=[
                    'Credit Card', 'PayPal', 'Bank Transfer', 'Cash'
                ])
            }
            row['total_amount'] = row['price'] * row['quantity']
            data.append(row)
        
        return pd.DataFrame(data)
    
    def generate_healthcare_dataset(self, n_samples=150):
        """Generate healthcare patient dataset."""
        data = []
        for i in range(n_samples):
            row = {
                'patient_id': f"P{self.fake.random_number(digits=6)}",
                'age': self.fake.random_int(0, 90),
                'gender': self.fake.random_element(elements=['M', 'F', 'O']),
                'diagnosis': self.fake.random_element(elements=[
                    'Hypertension', 'Diabetes', 'Asthma', 'Arthritis', 'Migraine'
                ]),
                'treatment_duration': self.fake.random_int(1, 365),
                'hospital': self.fake.company(),
                'doctor': self.fake.name(),
                'admission_date': self.fake.date_between(start_date='-2y', end_date='today'),
                'severity': self.fake.random_element(elements=['Low', 'Medium', 'High', 'Critical']),
                'cost': round(self.fake.pyfloat(left_digits=4, right_digits=2, positive=True) * 1000, 2)
            }
            data.append(row)
        
        return pd.DataFrame(data)
    
    def generate_comprehensive_report(self, datasets):
        """Generate comprehensive analysis report for all datasets."""
        report = {
            'generation_timestamp': datetime.now().isoformat(),
            'total_datasets': len(datasets),
            'dataset_summaries': {}
        }
        
        for domain, df in datasets.items():
            report['dataset_summaries'][domain] = {
                'shape': df.shape,
                'columns': list(df.columns),
                'numeric_columns': list(df.select_dtypes(include=[np.number]).columns),
                'categorical_columns': list(df.select_dtypes(include=['object']).columns),
                'memory_usage_mb': round(df.memory_usage(deep=True).sum() / 1024 / 1024, 3)
            }
        
        return report

try:
    factory = DatasetFactory()
    
    # Generate multiple domain datasets
    datasets = {}
    datasets['ecommerce'] = factory.generate_ecommerce_dataset(200)
    datasets['healthcare'] = factory.generate_healthcare_dataset(150)
    
    print(f"Generated {len(datasets)} domain-specific datasets")
    
    # Analyze each dataset
    for domain, df in datasets.items():
        print(f"\n=== {domain.upper()} Dataset Analysis ===")
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        
        # Show sample data
        print(f"Sample data (first 2 rows):")
        print(df.head(2))
        
        # Basic statistics for numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            print(f"\nNumeric column statistics:")
            print(df[numeric_cols].describe())
    
    # Generate comprehensive report
    report = factory.generate_comprehensive_report(datasets)
    
    print(f"\n=== Comprehensive Dataset Report ===")
    print(f"Generated at: {report['generation_timestamp']}")
    print(f"Total datasets: {report['total_datasets']}")
    
    for domain, summary in report['dataset_summaries'].items():
        print(f"\n{domain.upper()} Summary:")
        print(f"  Shape: {summary['shape']}")
        print(f"  Numeric features: {len(summary['numeric_columns'])}")
        print(f"  Categorical features: {len(summary['categorical_columns'])}")
        print(f"  Memory usage: {summary['memory_usage_mb']} MB")
    
    # Cross-domain analysis
    total_rows = sum(df.shape[0] for df in datasets.values())
    total_columns = sum(df.shape[1] for df in datasets.values())
    
    print(f"\n=== Cross-Domain Statistics ===")
    print(f"Total generated records: {total_rows}")
    print(f"Total feature columns: {total_columns}")
    print(f"Average dataset size: {total_rows / len(datasets):.0f} rows")
    
    # Export sample datasets to different formats
    sample_export = {
        'csv_exported': True,
        'json_exported': True,
        'formats_available': ['CSV', 'JSON', 'Parquet', 'Excel']
    }
    print(f"\nDataset export capabilities: {sample_export['formats_available']}")
    
except Exception as e:
    print(f"Error in advanced dataset generation: {e}")
    raise AssertionError(
        f"faker_datasets example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 completed")